In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.metrics import evaluate

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [3]:
ratings = pd.read_csv(
    "../data/ml-1m/ratings.dat",
    sep="::",
    names=["user_id", "movie_id", "rating", "timestamp"],
    engine="python",
)

ratings_sorted = ratings.sort_values("timestamp").reset_index(drop=True)
cutoff = int(len(ratings_sorted) * 0.8)
train = ratings_sorted[:cutoff]
test = ratings_sorted[cutoff:]

train_users = set(train["user_id"])
test_warm = test[test["user_id"].isin(train_users)]

print(f"Train: {len(train)}, Test (warm): {len(test_warm)}")

Train: 800167, Test (warm): 104540


In [4]:
user_ids = train["user_id"].unique()
movie_ids = train["movie_id"].unique()

user_to_idx = {uid: i for i, uid in enumerate(user_ids)}
movie_to_idx = {mid: i for i, mid in enumerate(movie_ids)}

n_users = len(user_to_idx)
n_movies = len(movie_to_idx)
print(f"n_users: {n_users}, n_movies: {n_movies}")

n_users: 5400, n_movies: 3662


In [5]:
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_movies, k=50):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, k)
        self.movie_factors = nn.Embedding(n_movies, k)
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        self.global_bias = nn.Parameter(torch.tensor(0.0))

    def forward(self, user, movie):
        p_u = self.user_factors(user)
        q_m = self.movie_factors(movie)
        b_u = self.user_bias(user).squeeze()
        b_m = self.movie_bias(movie).squeeze()
        dot = (p_u * q_m).sum(dim=1)
        return self.global_bias + b_u + b_m + dot

In [6]:
model = MatrixFactorization(n_users, n_movies, k=50)

test_users = torch.tensor([0, 1, 2, 3])
test_movies = torch.tensor([10, 20, 30, 40])

preds = model(test_users, test_movies)
print(preds.shape)
print(preds)

torch.Size([4])
tensor([-15.4584,  10.7823,  15.1397,   2.5327], grad_fn=<AddBackward0>)


In [7]:
train_user_idx = torch.tensor(train["user_id"].map(user_to_idx).values, dtype=torch.long)
train_movie_idx = torch.tensor(train["movie_id"].map(movie_to_idx).values, dtype=torch.long)
train_ratings = torch.tensor(train["rating"].values, dtype=torch.float)

print(train_user_idx.shape, train_movie_idx.shape, train_ratings.shape)

torch.Size([800167]) torch.Size([800167]) torch.Size([800167])


In [8]:
model = MatrixFactorization(n_users, n_movies, k=50)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)

In [9]:
n_epochs = 5
batch_size = 1024
n_train = len(train_ratings)

for epoch in range(n_epochs):
    perm = torch.randperm(n_train)

    total_loss = 0.0
    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]
        u = train_user_idx[idx]
        m = train_movie_idx[idx]
        r = train_ratings[idx]

        preds = model(u, m)
        loss = loss_fn(preds, r)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(idx)

    avg_loss = total_loss / n_train
    print(f"Epoch {epoch+1}: MSE = {avg_loss:.4f}")

Epoch 1: MSE = 20.8226
Epoch 2: MSE = 2.7192
Epoch 3: MSE = 1.1790
Epoch 4: MSE = 0.8486
Epoch 5: MSE = 0.7388


In [10]:
model.eval()

idx_to_movie = {i: mid for mid, i in movie_to_idx.items()}

train["user_idx"] = train["user_id"].map(user_to_idx)
train["movie_idx"] = train["movie_id"].map(movie_to_idx)
seen_by_user = train.groupby("user_idx")["movie_idx"].apply(set).to_dict()

all_movie_idx = torch.arange(n_movies)

def recommend_mf(user_idx, k=10):
    with torch.no_grad():
        users = torch.full((n_movies,), user_idx, dtype=torch.long)
        scores = model(users, all_movie_idx)
    ranked = torch.argsort(scores, descending=True).tolist()
    seen = seen_by_user.get(user_idx, set())
    recs = [idx_to_movie[i] for i in ranked if i not in seen][:k]
    return recs

In [11]:
warm_user_ids = list(test_warm["user_id"].unique())

recs_mf = {}
for uid in warm_user_ids:
    user_idx = user_to_idx[uid]
    recs_mf[uid] = recommend_mf(user_idx, k=10)

relevant_ratings = test_warm[test_warm["rating"] >= 4]
relevant_by_user = relevant_ratings.groupby("user_id")["movie_id"].apply(set).to_dict()

results_mf = evaluate(recs_mf, relevant_by_user, k=10)
print(f"Precision@10: {results_mf['precision']:.4f}")
print(f"Recall@10:    {results_mf['recall']:.4f}")
print(f"NDCG@10:      {results_mf['ndcg']:.4f}")

Precision@10: 0.0138
Recall@10:    0.0030
NDCG@10:      0.0147


In [12]:
sample = recommend_mf(user_to_idx[warm_user_ids[0]], k=10)
print("Recommended movie IDs:", sample)

counts = train["movie_id"].value_counts()
print("Train rating counts for these:", [int(counts.get(m, 0)) for m in sample])

Recommended movie IDs: [np.int64(964), np.int64(3676), np.int64(2983), np.int64(3341), np.int64(3078), np.int64(2732), np.int64(9), np.int64(214), np.int64(2493), np.int64(3618)]
Train rating counts for these: [42, 162, 67, 147, 117, 123, 80, 18, 21, 213]


In [13]:
for uid in warm_user_ids:
    if uid in relevant_by_user and len(relevant_by_user[uid]) > 0:
        print("User:", uid)
        print("Relevant (rated >=4 in test):", relevant_by_user[uid])
        print("We recommended:", recs_mf[uid])
        print("Overlap:", set(recs_mf[uid]) & relevant_by_user[uid])
        

User: 1875
Relevant (rated >=4 in test): {1409, 260, 3206, 1287, 648, 2311, 10, 11, 2316, 1291, 2571, 3471, 780, 919, 1303, 539, 924, 1947, 2971, 674, 932, 1573, 3494, 2470, 1320, 1066, 938, 1196, 1198, 1200, 945, 1204, 3638, 2871, 1206, 1721, 1722, 1081, 1597, 2621, 2366, 1088, 2367, 1214, 2115, 2628, 1220, 329, 1097, 1356, 1101, 2640, 2641, 1240, 2393, 3675, 1374, 3168, 736, 2529, 1376, 2916, 1381, 1250, 2407, 750, 3827, 3061, 1270, 3703, 2942, 2303}
We recommended: [np.int64(964), np.int64(3676), np.int64(2983), np.int64(3341), np.int64(3078), np.int64(2732), np.int64(9), np.int64(214), np.int64(2493), np.int64(3618)]
Overlap: set()
User: 635
Relevant (rated >=4 in test): {1, 2565, 3599, 3088, 3089, 529, 2068, 3604, 2070, 3095, 2583, 3613, 1054, 2080, 34, 36, 2088, 41, 2099, 1079, 58, 1083, 1096, 593, 3163, 3675, 3678, 608, 1635, 3176, 1131, 1132, 1660, 1150, 3199, 3200, 2182, 3724, 1177, 2208, 1185, 1187, 164, 1188, 1704, 1711, 1732, 1221, 2757, 199, 1225, 1227, 3789, 1230, 1231, 1